In [65]:
import torch
import json
import numpy as np
from sklearn import model_selection
from ase.formula import Formula

In [25]:
with open ("../data/raw/phonon_materials.json") as f:
    data = json.load(f)

In [55]:
labels = np.array([1 if entry['has_gap'] else 0 for entry in data],dtype=int)

In [43]:
train_idx, test_validation_idx = model_selection.train_test_split(np.arange(len(labels)), test_size=0.30, stratify=labels, random_state=0)
test_idx, validation_idx = model_selection.train_test_split(test_validation_idx, test_size=0.5, stratify=labels[test_validation_idx], random_state=0)



In [63]:
formula_list = np.array([entry['formula'] for entry in data],dtype=str)

In [68]:
form_dict_list = [Formula(f).count() for f in formula_list]

In [70]:
key_list = []

for entry in form_dict_list:
    for key in entry.keys():
        if key not in key_list:
            key_list.append(key)

In [79]:

key_dict = dict.fromkeys(key_list,0)

In [96]:
features = []
for entry in form_dict_list:
    features.append(np.array(list((key_dict | entry).values())))

In [97]:
features = np.array(features)

In [98]:

features_train = features[train_idx]
features_test = features[test_idx]
features_validate = features[validation_idx]

labels_train = labels[train_idx]
labels_test = labels[test_idx]
labels_validate = labels[validation_idx]

In [99]:
features_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 2, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(1058, 63))

In [108]:
X_train = torch.tensor(features_train,dtype=torch.float32)
y_train = torch.tensor(labels_train,dtype=torch.float32).reshape(-1,1)

X_test = torch.tensor(features_test,dtype=torch.float32)
y_test = torch.tensor(labels_test,dtype=torch.float32).reshape(-1,1)

In [114]:
import torch.nn as nn
import copy
import numpy as np
import torch
import torch.optim as optim
import tqdm

class First_Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(63, 180)
        self.relu = nn.ReLU()
        self.output = nn.Linear(180,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,x):
        x = self.sigmoid(self.output(self.relu(self.hidden(x))))
        return x


In [ ]:
def model_train(model, X_train, y_train, X_test, y_test):

    loss_fn = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(),lr=0.0001)

    n_epochs = 250
    batch_size = 10
    batch_start = torch.arange(0, len(X_train), batch_size)

    best_acc = -np.inf
    best_weights = None

    for epoch in range(n_epochs):
        model.train()
        with tqdm.tqdm(batch_start, unit="batch", mininterval=0, disable=True) as bar:
            bar.set_description(f"Epoch {epoch}")
            for start in bar:
                X_batch = X_train[start:start+batch_size]
                y_batch = y_train[start:start+batch_size]

                y_pred = model(X_batch)
                
                loss = loss_fn(y_pred, y_batch)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                acc = (y_pred.round() == y_batch).float().mean()
                bar.set_postfix(
                    loss=float(loss),
                    acc=float(acc)
                )
        # evaluate accuracy at end of each epoch
        model.eval()
        y_pred = model(X_test)
        
        acc = (y_pred.round() == y_test).float().mean()
        acc = float(acc)
        if acc > best_acc:
            best_acc = acc
            best_weights = copy.deepcopy(model.state_dict())

    print(y_pred.round()[:], y_test)
    # restore model and return best accuracy
    model.load_state_dict(best_weights)
    return best_acc

In [125]:
model_train(First_Model(),X_train, y_train, X_test, y_test)

tensor([[0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
      

0.7973568439483643